1. Install the Vertex AI SDK: Open a terminal window and enter the command below. You can also [install it in a virtualenv](https://googleapis.dev/python/aiplatform/latest/index.html)

In [3]:
import vertexai

vertexai.init(project="rhazes-research", location="us-central1")

In [4]:
from google.colab import auth

auth.authenticate_user()

In [5]:
from google import genai
from google.genai import types
import base64

def generate(prompt):
  client = genai.Client(
      vertexai=True,
      project="rhazes-research",
      location="global",
  )


  model = "gemini-2.5-flash"
  contents = [
    types.Content(
      role="user",
      parts=[
        types.Part.from_text(text=prompt)
      ]
    ),
  ]

  generate_content_config = types.GenerateContentConfig(
    temperature = 1,
    top_p = 1,
    seed = 0,
    max_output_tokens = 65535,
    safety_settings = [types.SafetySetting(
      category="HARM_CATEGORY_HATE_SPEECH",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_DANGEROUS_CONTENT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
      threshold="OFF"
    ),types.SafetySetting(
      category="HARM_CATEGORY_HARASSMENT",
      threshold="OFF"
    )],
    thinking_config=types.ThinkingConfig(
      thinking_budget=-1,
    ),
  )

  res = client.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
    )

  return res

chunks = generate('hey this is a test how are you')
# chunks now contains a list of the text from each chunk.


In [6]:
print(chunks.candidates[0].content.parts[0].text)

Hello! As an AI, I don't have feelings or a physical state, so I don't experience "being" well or unwell. However, I'm fully operational and ready to assist you!

How can I help you today?


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

global result_df, prompt_no_example, generate_content_config

result_df = pd.read_csv('/content/drive/MyDrive/MIMIC-IV_input.csv', index_col=0)

In [8]:
prompt_no_example = '''You are an expert diagnostician machine for use by doctors. If the user input is not patient data, you politely decline the request. Please suggest diagnoses and conditions, followed by the evidence points supporting each diagnosis in the form of bullet points. Include previous diagnoses and pertinent information about the patient's medical history (if any). Pay close attention to all the history and investigations provided.  Put asterisks around the diagnoses to highlight them. Give each evidence points as a separate bullet point beneath the diagnosis. Include in your evidence points any relevant clinical scores that can be calculated from the information I have given. Do not explain the evidence points, only state them. For every diagnosis you list, if there are alternative differentials possible, state the most likely three in a bullet point beneath the evidence points (you do not need to state the evidence supporting them - you only need to do that for the main diagnoses). For the main diagnoses, give only confirmed diagnoses and evidence points that can be inferred solely based on the information I have given - do not use any other information. Only give me the information I have asked for - do not give me any other information. Do not give me any introductions or conclusions, safety instructions, or safety warnings. Use British English.

                                To illustrate how the information should be presented:

                                *MAIN DIAGNOSIS 1 AS HEADING*
                                evidence points to support MAIN DIAGNOSIS 1
                                The final bullet point is alternative differentials to consider: alternative 1, alternative 2, alternative 3

                                *MAIN DIAGNOSIS 2 AS HEADING*
                                evidence points to support MAIN DIAGNOSIS 2
                                The final bullet point is alternative differentials to consider: alternative 1, alternative 2, alternative 3

                                and so on...

Before finalising your answer check if you haven't missed any abnormal data points and hence any diagnoses or alternative differentials that could be made based on them. If you did, add them to your reply. If two diagnoses are commonly caused by the same underlying disease, have them under one header, which is the underlying disease.

Patient data:\n'''


In [9]:
generate_content_config = types.GenerateContentConfig(
    temperature=1,
    top_p=1,
    seed=0,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF")
    ],
    thinking_config=types.ThinkingConfig(
        thinking_budget=-1,
    ),
)

In [10]:
import asyncio
from google import genai
from google.genai import types
import pandas as pd
from tqdm import tqdm
import nest_asyncio

nest_asyncio.apply()

#moved outside the async function
client = genai.Client(
    vertexai=True,
    project="rhazes-research",
    location="global",
)

async def generate(prompt):
    model = "gemini-2.5-flash"
    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=prompt)]
        ),
    ]

    res = await client.aio.models.generate_content(
    model = model,
    contents = contents,
    config = generate_content_config,
    )

    return res.candidates[0].content.parts[0].text

async def process_row(i):
    response = await generate(  # Await the generate function
        prompt_no_example + result_df.iloc[i]['GPT_input']
    )
    hadm_id = result_df.index[i]
    return hadm_id, ''.join(response) # Return both hadm_id and the response

async def get_diagnoses(indices):
    tasks = []
    for i in indices:
        tasks.append(process_row(i))
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
result_df['GPT-Diagnoses'] = result_df['GPT-Diagnoses'].astype('str')
result_df['GPT-Eval'] = result_df['GPT-Eval'].astype('str')
repeat = []

limit = 1000
chunk_size = 5

def generate_indices(limit, chunk_size):
  """Generates lists of indices in chunks."""
  start = 0
  while start < limit:
    end = min(start + chunk_size, limit)  # Ensure end doesn't exceed the limit
    yield list(range(start, end))
    start = end

def get_total_iterations(limit, chunk_size):
  """Calculates the total number of iterations."""
  return (limit + chunk_size - 1) // chunk_size

total_iterations = get_total_iterations(limit, chunk_size)

# Example usage:
for indices in tqdm(generate_indices(limit, chunk_size), total=total_iterations):
    try:
        results = asyncio.run(get_diagnoses(indices))
        for result in results:
          hadm_id = result[0]
          content = result[1]
          result_df.loc[hadm_id, 'GPT-Diagnoses'] = content
    except Exception as e:
        print('Error happened at iteration i: ' + str(indices[0]/chunk_size))
        repeat.append(i for i in indices)
        print(e)

 16%|█▌        | 32/200 [33:32<1:57:25, 41.94s/it]

Error happened at iteration i: 31.0
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}


 22%|██▎       | 45/200 [45:17<1:48:38, 42.05s/it]

Error happened at iteration i: 44.0
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}


 29%|██▉       | 58/200 [57:18<1:48:19, 45.77s/it]

...

 30%|███       | 60/200 [58:23<1:23:56, 35.98s/it]

Error happened at iteration i: 59.0
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}


 93%|█████████▎| 186/200 [3:02:52<09:48, 42.02s/it]

Error happened at iteration i: 185.0
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}


100%|██████████| 200/200 [3:17:27<00:00, 59.24s/it]


In [ ]:
result_df.to_csv('/content/drive/MyDrive/Gemini25ProdDiagnoses.csv')

In [11]:
import pandas as pd
import numpy as np
from tqdm import tqdm

result_df = pd.read_csv('/content/drive/MyDrive/Gemini25ProdDiagnoses.csv', index_col=0)

In [25]:
result_df.iloc[220:]

,hadm_id,diagnoses,GPT_input,GPT-Diagnoses,GPT-Eval
220,25254429,1:Other encephalopathy\n2:Acute kidney failure...,Blood report: \nThe patient stayed in the hosp...,NaN,NaN
221,23517942,"1:Acute kidney failure, unspecified\n2:Hyperos...",Blood report: \nThe patient stayed in the hosp...,NaN,NaN
222,29971278,1:Malignant neoplasm of splenic flexure\n2:Hyp...,Blood report: \nThe patient stayed in the hosp...,NaN,NaN
223,27775427,1:Other specified acquired deformity of head\n...,Blood report: \nThe patient stayed in the hosp...,NaN,NaN
224,26082075,"1:Acute kidney failure, unspecified\n2:Acidosi...",Blood report: \nThe patient stayed in the hosp...,NaN,NaN
...,...,...,...,...,...
995,27134177,1:Other chest pain\n2:Atherosclerotic heart di...,Blood report: \nThe patient stayed in the hosp...,Patient's medical history:\n* Coronary Arter...,NaN
996,25290618,"1:Pneumonia, organism unspecified\n2:Acute pan...",Blood report: \nThe patient stayed in the hosp...,*Pneumonia (Right Upper Lobe)*\n* Patient pr...,NaN
997,29117810,1:Coronary atherosclerosis of native coronary ...,Blood report: \nThe patient stayed in the hosp...,*POSTOPERATIVE BLEEDING AND COAGULOPATHY*\n* ...,NaN
998,26085629,"1:Pneumonia, organism unspecified\n2:Liver rep...",Blood report: \nThe patient stayed in the hosp...,*Multi-focal Pneumonia*\n* Nodular opacity i...,NaN


In [16]:
# prompt: list of list of 5 consecutive numbers from 0 to 999 starting[ [0,1,2,3,4], [5,6,7,8,9], etc ]

list_of_lists = [[i for i in range(j, j + 5)] for j in range(0, 1000, 5)]
print(list_of_lists[31])
print(list_of_lists[44])
print(list_of_lists[57])
print(list_of_lists[59])
print(list_of_lists[185])


[155, 156, 157, 158, 159]
[220, 221, 222, 223, 224]
[285, 286, 287, 288, 289]
[295, 296, 297, 298, 299]
[925, 926, 927, 928, 929]


In [29]:
missed = np.array(list_of_lists)[[31, 44, 57, 59, 185]].tolist()

In [30]:
missed

[[155, 156, 157, 158, 159],
 [220, 221, 222, 223, 224],
 [285, 286, 287, 288, 289],
 [295, 296, 297, 298, 299],
 [925, 926, 927, 928, 929]]

In [32]:
for indices in tqdm(missed):
    print(indices)
    try:
        results = asyncio.run(get_diagnoses(indices))
        for result in results:
          hadm_id = result[0]
          content = result[1]
          result_df.loc[hadm_id, 'GPT-Diagnoses'] = content
    except Exception as e:
        print('Error happened at iteration')
        repeat.append(i for i in indices)
        print(e)

  0%|          | 0/5 [00:00<?, ?it/s]

[155, 156, 157, 158, 159]


 20%|██        | 1/5 [00:52<03:30, 52.60s/it]

[220, 221, 222, 223, 224]


 40%|████      | 2/5 [01:52<02:50, 56.97s/it]

...

 60%|██████    | 3/5 [02:52<01:56, 58.37s/it]

[295, 296, 297, 298, 299]


 80%|████████  | 4/5 [03:44<00:55, 55.66s/it]

[925, 926, 927, 928, 929]


100%|██████████| 5/5 [04:46<00:00, 57.21s/it]


In [33]:
result_df['GPT-Diagnoses'].isna().sum()

np.int64(0)

In [34]:
result_df.to_csv('/content/drive/MyDrive/Gemini25ProdDiagnoses.csv')

In [35]:
result_df.to_csv('Gemini25ProdDiagnoses.csv')